## Rappresentazione dei circuiti e calcolo dell'equivalenza
Si definisce la classe Circuit in grado di rappresentare circuiti a n qubit e m time-steps come stati quantistici, con associata la classe CircuitShape che gestisce i possibili timesteps dato l'insieme dei gate disponibili ed il numero di qubit

In [62]:
import numpy as np
import csv
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Operator

class CircuitState:
    def __init__(self,shape,steps):  
        self._shape = shape
        
        simplified_steps = [steps[0].copy()]
        for step in steps[1:]:
            if step != self._shape._getID():
                merged = []
                for i in range(len(step)):
                    gate_to_merge = []
                    if simplified_steps[-1][i] == "ID":
                        if step[i][-1] == "c":
                            for j in range(i+1,len(step)):
                                if j not in merged and step[j][:-1] == step[i][:-1] and step[j][-1] == "t":
                                    if simplified_steps[-1][j] == "ID":
                                        gate_to_merge.append(i)
                                        gate_to_merge.append(j)
                                        merged.append(i)
                                        merged.append(j)
                                    break
                        elif step[i] != "ID" and not step[i][-1] == "t":
                            gate_to_merge.append(i)
                            merged.append(i)
                        if len(gate_to_merge) != 0:
                            k = len(simplified_steps)
                            while(k>0 and all(simplified_steps[k-1][l]=="ID" for l in gate_to_merge)):
                                k-=1
                            for l in gate_to_merge:
                                simplified_steps[k][l] = step[l]
                new_step = ["ID" if i in merged else step[i] for i in range(len(step)) ]
                if new_step != self._shape._getID():
                    simplified_steps.append( new_step ) 
        
        self._state = np.zeros((self._shape.get_n_steps()**len(simplified_steps)))
        index = 0
        for i in range(len(simplified_steps)):
            index += self._shape.get_step_index(simplified_steps[i]) * self._shape.get_n_steps()**(len(simplified_steps)-1-i)
        self._state[index] = 1.

    def get_n_qubits(self):
        return self._shape.get_n_qubits()
    def get_n_steps(self):
        return round(np.emath.logn(self._shape.get_n_steps(), self._state.shape[0]))
    
    # converts quantum state to human readable circuit
    # only works if the state is classical
    def get_steps(self):
        if not self.is_classical(): raise Exception("The state is not classical")
        full_index = self._to_base(np.where(self._state == 1.)[0][0], self._shape.get_n_steps())
        return [self._shape._steps[i] for i in full_index]
    
    def is_classical(self):
        return len(np.where(self._state == 1)[0]) == 1
    
    def _to_base(self,number, base):
        result = np.zeros((self.get_n_steps()),dtype=np.intc)
        i=0
        while number:
            result[i] = number % base
            number //= base
            i+=1
        return result[::-1]

    def get_state(self):
        return self._state.copy()
    
    def getID(shape):
        return CircuitState(shape,[shape._getID()])

    # saves the circuit in a csv file
    def to_csv(self,filename):
        with open(filename, 'w', newline='') as fout:
            writer = csv.writer(fout, delimiter=",")
            for step in self.get_steps():
                writer.writerow( step )
        
    #static method to create a circiut from a csv file
    def load_csv(filename,shape):
        with open(filename, 'r', newline='') as fin:
            reader = csv.reader(fin, delimiter=",")
            steps = [row for row in reader]
            return CircuitState(shape,steps)
    load_csv = load_csv
    
    # returns the corresponding qiskit.circuit.QuantumCircuit
    def to_qiskit(self):
        if not self.is_classical(): raise Exception("The state is not classical")
        
        steps = self.get_steps()
        circuit = QuantumCircuit(self._shape.get_n_qubits())
        for step in steps:
            taken = []
            for i in range(len(step)):
                    if step[i][-1] == "c":
                        for j in range(i+1,len(step)):
                            if j not in taken and step[j][:-1] == step[i][:-1] and step[j][-1] == "t":
                                if step[i][:-1] == "S":
                                    circuit.swap(i,j)
                                elif step[i][:-1] == "CZ":
                                    circuit.cz(i,j)
                                taken.append(j)
                                break #breaks when the gate is close
        return circuit

    # draws the circuit using qiskit.circuit.QuantumCircuit.draw() method
    def draw(self,mpl=False):
        if not mpl:
            return self.to_qiskit().draw(output='text')
        else:
            return self.to_qiskit().draw("mpl")
        
    def equals(self,circuit):
        return np.array_equal(self._state,circuit._state)
    # return true if the circuit is equivalent to ID
    def isID(self):
        U = Operator(self.to_qiskit()).to_matrix()
        return np.linalg.norm(U/U[0,0] - np.identity( 2**self.get_n_qubits() )) < 1e-12
    # returns true if c1 and c2 (Cricuit obejcts) implement the same algorithm
    def equivalent(c1,c2):
        U = Operator(c1.to_qiskit().compose(c2.to_qiskit().inverse())).to_matrix()
        return np.linalg.norm(U/U[0,0] - np.identity( 2**c1.get_n_qubits() )) < 1e-12
    equivalent = equivalent

In [63]:
class CircuitShape:
    def __init__(self,n_qubits,gates):
        self._n_qubits = n_qubits
        self._steps = []
        self._find_steps(gates,[],0)
        
    # recursively finds possible timesteps given a set of gates and a number of qubits    
    def _find_steps(self,gates,step,length):
        if length == self._n_qubits:
            self._steps.append(step)
        else:
            for gate_length in gates.keys():
                for gate in gates[gate_length]:
                    if length+gate_length <= self._n_qubits:
                        new_step = step.copy()
                        for block in gate:
                            new_step.append(block)
                        self._find_steps(gates,new_step,length+gate_length)
    
    # returns quantum state from single step list
    def state_of_step(self,step):
        state = np.zeros((len(self._steps)),dtype=np.float64)
        state[self._steps.index(step)] = 1.
        return state
    # return step from single step quantum state
    def step_of_state(self,state):
        return self._steps[np.where(state==1.)[0][0]]
    def index_of_state(self,state):
        return np.where(state==1.)[0][0]
    def get_n_qubits(self):
        return self._n_qubits
    def get_step_index(self,step):
        return self._steps.index(step)
    def get_n_steps(self):
        return len(self._steps)
    def _getID(self):
        return self._steps[0]
    
    def get_steps(self):
        return self._steps.copy()
    
    def print_steps(self,auto_print=True):
        out = ""
        for i in range(len(self._steps)):
            out += str(i) + ": " + str(self._steps[i]) + "\n"
        if auto_print: print(out)
        return out
    
    def find_equivalences(gates,n_qubits,d_max,dir_out):
        shape = CircuitShape(n_qubits, gates)
        
        for d in range(2,d_max+1):
            # creates a list of not trivially equivalent circuits of effective d timesteps
            # and a list of id equivalent circuits of effective d timesteps
            actual_circuits = []
            IDequivalent = []
            available_steps = shape.get_steps()

            ids = list(it.product(range(len(available_steps)), repeat=d))
            for i in range(len(ids)):
                c = CircuitState(shape, [available_steps[k] for k in ids[i]])
                if c.get_n_steps() == d and c not in IDequivalent:
                    if c.isID():
                        IDequivalent.append(c)
                    elif all(not c.equals(c1) for c1 in actual_circuits):
                        actual_circuits.append(c)
        
        filename = dir_out + "/eq_d=" + str(d) + ".csv"
        with open(filename, 'w', newline='') as fout:
            writer = csv.writer(fout, delimiter="\t")
            writer.writerow(["n_qubits:" +str(n_qubits)])
            writer.writerow(["gates:"])
            for key in gates.keys():
                writer.writerow(gates[key])
            writer.writerow(["equivalences:"])
            for i in range(len(actual_circuits)-1):
                for j in range(i+1,len(actual_circuits)):
                    if CircuitState.equivalent(actual_circuits[i],actual_circuits[j]):
                        writer.writerow( [shape.get_step_index(step) for step in actual_circuits[i].get_steps()]
                                        + [shape.get_step_index(step) for step in actual_circuits[j].get_steps()])
        
        filename = dir_out + "/id_d=" + str(d) + ".csv"
        with open(filename, 'w', newline='') as fout:
            writer = csv.writer(fout, delimiter="\t")
            writer.writerow(["n_qubits:" +str(n_qubits)])
            writer.writerow(["gates:"])
            for key in gates.keys():
                writer.writerow(gates[key])
            writer.writerow(["id equivalences:"])
            for c in IDequivalent:
                writer.writerow([shape.get_step_index(step) for step in c.get_steps()])

Le due classi una volta scritte e testate vengono aggiunte al modulo QuantumCompiling

In [64]:
#from QuantumCompiling import Circuit, CircuitShape

Test del calcolo dei possibili time-step con la classe CircuitShape

In [65]:
n_qubits = 4
gates = { 
    1 : [ ["ID"]],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    ,3 : [ ["Sc","ID","St"] ]
    ,4 : [ ["Sc","Sc","St","St"] ] 
}

shape = CircuitShape(n_qubits,gates)
shape.print_steps(); 

0: ['ID', 'ID', 'ID', 'ID']
1: ['ID', 'ID', 'Sc', 'St']
2: ['ID', 'ID', 'CZc', 'CZt']
3: ['ID', 'Sc', 'St', 'ID']
4: ['ID', 'CZc', 'CZt', 'ID']
5: ['ID', 'Sc', 'ID', 'St']
6: ['Sc', 'St', 'ID', 'ID']
7: ['Sc', 'St', 'Sc', 'St']
8: ['Sc', 'St', 'CZc', 'CZt']
9: ['CZc', 'CZt', 'ID', 'ID']
10: ['CZc', 'CZt', 'Sc', 'St']
11: ['CZc', 'CZt', 'CZc', 'CZt']
12: ['Sc', 'ID', 'St', 'ID']
13: ['Sc', 'Sc', 'St', 'St']



Test della classe Circuit: calcolo dello stato dato un circuito human readable e operazione inversa

In [73]:
steps = [
    ['Sc', 'St',"ID","ID"],
    ["ID",'Sc', 'St',"ID"],
    ["ID","ID","ID","ID"],
    ['Sc', 'St', 'Sc', 'St']
]

# steps3 = [
#     ["ID","ID","ID"],
#     ["Sc","St","ID"],
#     ["CZc","CZt","ID"],
#     ["ID","CZc","CZt"]
# ]

circuit = CircuitState(shape,steps)

# circuit._state[6] = 0.5
# circuit._state[0] = 0.5

print("Stato:", circuit.get_state())
print("Il circuito è composto da:", circuit.get_n_steps(),"step (le identità vengono correttamente semplificate)")
print("Conversione da stato a steps:",circuit.get_steps())
print("Lo stato è classico:",circuit.is_classical())

Stato: [0. 0. 0. ... 0. 0. 0.]
Il circuito è composto da: 3 step (le identità vengono correttamente semplificate)
Conversione da stato a steps: [['Sc', 'St', 'ID', 'ID'], ['ID', 'Sc', 'St', 'ID'], ['Sc', 'St', 'Sc', 'St']]
Lo stato è classico: True


Test del metodo to qiskit per convertire un oggeto Circuit in un oggetto qiskit.circuit.QuantumCircuit

In [67]:
circuit.to_qiskit().draw()

q_0: ─X─────X─
      │     │ 
q_1: ─X──X──X─
         │    
q_2: ────X──X─
            │ 
q_3: ───────X─

Verifica della semplificazione dei circuiti

In [82]:
n_qubits = 6
gates = { 
    1 : [ ["ID"],["H"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    ,3 : [ ["Sc","ID","St"] ]
    ,4 : [ ["Sc","Sc","St","St"] ] 
}

shape1 = CircuitShape(n_qubits,gates)

steps = [
    ["CZc","CZt","ID","ID","ID","ID"],
    ["ID" ,"ID", "Sc","St","ID","ID"],
    ['ID' ,'Sc', 'ID','St',"ID","ID"],
    ['H'  ,'ID', 'ID','ID',"Sc","St"],
    ["ID" ,"ID","Sc" ,"ID","St","H"]
    
]

circuit = CircuitState(shape1,steps)
circuit.get_steps()

[['CZc', 'CZt', 'Sc', 'St', 'Sc', 'St'], ['H', 'Sc', 'Sc', 'St', 'St', 'H']]

Verifica della possibilità di creare un circuito identità

In [69]:
steps = [
    ["ID","ID","ID","ID"],
    ["ID","ID","ID","ID"],
    ["ID","ID","ID","ID"]
]

circuit = CircuitState(shape,steps)
circuit.get_steps()

[['ID', 'ID', 'ID', 'ID']]

Test dei metodi di caricamento/salvataggio dei circuiti da/su csv

In [70]:
c1 = CircuitState.load_csv("csv/c1.csv",shape)
print("Circuito caricato:", c1.draw())

#salvataggio su csv
c1.to_csv("csv/c1_out.csv")
print("Circuito salvato:", CircuitState.load_csv("csv/c1_out.csv",shape).draw())

Circuito caricato:               
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
Circuito salvato:               
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              


Test del calcolo dell'equivalenza tra due circuiti: si utilizzano i metodi qiskit per calcolare se due circuiti sono o non sono equivalenti. In particolare si concatena il primo circuito passato come parametro con l'inverso del secondo, si calcola l'operatore U che realizza il circuito complessivo e se ne verifica l'uguaglianza con l'identità

In [71]:
c1 = CircuitState.load_csv("csv/c1.csv",shape)
c2 = CircuitState.load_csv("csv/c2.csv",shape)
c3 = CircuitState.load_csv("csv/c3.csv",shape)

print("Equivalenza c1-c2:",CircuitState.equivalent(c1,c2))
print("Equivalenza c1-c3:",CircuitState.equivalent(c1,c3))

print(c1.draw())
print(c2.draw())
print(c3.draw())

Equivalenza c1-c2: True
Equivalenza c1-c3: False
              
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
              
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
              
q_0: ─■───────
      │       
q_1: ─■──X──X─
         │  │ 
q_2: ────X──X─
              
q_3: ─────────
              


In [72]:
a = "a"
b = "b"
c = [a,b]
z = c[0]
z = "f"
print(c)

['a', 'b']
